> 可运行笔记：第 4 格会真调 DeepSeek，跑两次约 0.01 元。
> 不想花钱就把第 1 格换成 mock，第 3 格完全不用改。

# d02 · 2026-10-30（周五）· 报文 → 最小 loop

> 今天的命题只有一个：让 `invoke()` 把模型吐出的 `tool_calls` 拆成「工具名 + 参数字典」。
> 在看过真实报文之前不要写这行代码，否则你就是在猜格式。
> 交付 = 本文件填完 + `src/agent/loop.py` 能跑一遍 + `PROGRESS.md` 加一行。

---

## 0. 补 d01 欠的报文（25 min）

选**一家**，只读它的「直接示例」那一节，读到你能抄出响应 JSON 就停（不读流式、不读并行调用、不读多轮）。

- DeepSeek：https://api-docs.deepseek.com/guides/function_calling
- Anthropic：https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview

### 请求里的 tools 定义（脱敏，去掉 key）

```json

```

### 响应里的 tool_calls（脱敏）

```json

```

### 三个细节（填完才往下走）

1. 模型决定「调哪个工具 / 传什么参数」，这两个信息分别落在 `______` 和 `______`，类型是字符串还是对象：`______`
2. 工具结果回灌时用什么角色 / 什么字段：`______`（这决定你下一轮 append 什么进 messages）
3. 我原以为 `______`，实际是 `______`

---

## 1. `invoke()` + `run()`（30 min，关掉 AI 写）

改 `src/agent/loop.py`。工具先写**假的**：`get_weather()` 返回一个 dict，`control_ac(temp)` 返回一个 dict。

写的时候自己盯这三个地方（V2 里已经埋了）：

- [ ] `except OSError` —— 你打算抛的是 OSError 吗？还是 `ValueError` / `KeyError`？抓不住的那个会让循环直接死
- [ ] `except` 之后 `fun_name` 绑定上了吗？异常路径走下来还能解引用吗
- [ ] 循环的**停止条件**在哪：跑满 max_steps？工具返回了「完成」？还是模型不再给 tool_calls？

### 我写的代码（贴进来）

### 跑一次的真实输出（终端原文，不是复述）

```

```

---

## 1.5 参考实现（读，不要抄）

按你的要求先给了完整实现。三份文件，都在 `src/` 下：

| 文件 | 行数 | 里面是什么 |
|---|---|---|
| `src/agent/loop.py` | 146 | 主循环 + 单个 tool_call 的执行。`# CORE` 5 处是骨架 |
| `src/llm/client.py` | 105 | 一次 HTTP 往返、重试、记账。标准库 urllib，没装 sdk |
| `src/llm/types.py` | 57 | 定价表、`Usage`、`ChatResult`、`LLMError` |
| `src/trace.py` | 54 | JSONL 落盘，字段固定成一张表 |
| `src/tools/builtin.py` | 119 | `Tool` = schema + handler 绑一起；3 个工具；车况真对象 `STATE` |

五个文件全部 ≤150 行——AGENTS.md §5 的硬约束。**第一次写 176 行、client 159 行，是被这条规则逼着拆的**：
trace 是横向能力（网关/评测/回放都要读），不该长在 loop 身上；价格表和 HTTP 也是两件事。
这个拆法本身就是要学的东西，不是形式主义。

**读的顺序**（别从头读，会淹）：先在 `loop.py` 里只找 `# CORE 1`～`# CORE 5` 这五个注释，把
「调模型 → 判停 → 逐个执行 → 执行 → 回灌」串成一段话；再看 `client.chat()` 里那个 payload
字典长什么样（`tools` 怎么传、`tool_choice` 什么用）；最后看 `Tool.schema()`——它就是模型看到的
那份 JSON，`description` 写得好不好直接决定模型调不调。

**关于 `notes/d02-reference/` 那 5 个文件**：它们按 `src/` 的包布局写 import（`from src.tools.builtin import ...`），而 `src/` 归你自己写、现在是空的，所以那 5 个文件**只读不跑**——它们是「怎么拆成 5 个文件、每个文件放什么」的样本。

能直接跑的是下面这 4 格自包含版本（不需要 `src/`，只要一个 key）。

### 下面这 4 格是**真能跑**的最小版（需要 `DEEPSEEK_API_KEY`）

依次执行：① 客户端 → ② 工具 → ③ 主循环 → ④ 跑一次。
三个 `# CORE` 注释就是你要默写的东西；去掉注释、去掉 print，剩下的才是 agent 本身。

In [ ]:
# ---- 1/4 最小 LLM 客户端（只用标准库）------------------------------------
# 工程版会加重试、记账、超时、换模型——那些在 notes/d02-reference/client.py。
import json, os, urllib.request

API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
BASE_URL = "https://api.deepseek.com/v1"
MODEL = "deepseek-chat"


def chat(messages, tools):
    if not API_KEY:
        raise RuntimeError("先在终端 export DEEPSEEK_API_KEY=sk-... 再启动 notebook")
    body = json.dumps({"model": MODEL, "messages": messages, "tools": tools,
                       "tool_choice": "auto", "temperature": 0}).encode()
    req = urllib.request.Request(
        f"{BASE_URL}/chat/completions", data=body,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {API_KEY}"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read())

In [ ]:
# ---- 2/4 车况真对象 + 工具（schema 和 handler 绑在一起）-------------------
STATE = {"ac_temp": 26, "ac_on": False, "speed_kmh": 0}


def control_ac(args):
    t = args.get("temperature")
    if isinstance(t, bool) or not isinstance(t, int):
        return {"ok": False, "error": {"code": "BAD_TYPE", "message": f"temperature 要整数，收到 {t!r}",
                                       "hint": '例如 {"temperature": 24}'}}
    if not 16 <= t <= 30:
        return {"ok": False, "error": {"code": "OUT_OF_RANGE", "message": f"{t} 超出 16~30",
                                       "hint": "给 16~30 之间的整数；用户没说数值就先问"}}
    STATE["ac_temp"], STATE["ac_on"] = t, True
    return {"ok": True, "data": dict(STATE)}


def get_vehicle_status(args):
    return {"ok": True, "data": dict(STATE)}


TOOLS = {
    "control_ac": {
        "schema": {"type": "function", "function": {
            "name": "control_ac",
            "description": ("设置车内空调目标温度并开启空调。只在用户给出 16~30 之间的具体数值时调用；"
                            "用户说「调凉一点」但没给数值时不要猜，先问清楚。"),
            "parameters": {"type": "object", "properties": {
                "temperature": {"type": "integer", "minimum": 16, "maximum": 30,
                                "description": "目标温度，摄氏度整数"}},
                "required": ["temperature"]}}},
        "handler": control_ac},
    "get_vehicle_status": {
        "schema": {"type": "function", "function": {
            "name": "get_vehicle_status",
            "description": "读取当前车况（车速/空调开关/目标温度）。只读。执行改动前先调用确认。",
            "parameters": {"type": "object", "properties": {}}}},
        "handler": get_vehicle_status},
}

In [ ]:
# ---- 3/4 主循环：agent 的全部骨架就是这一格 -------------------------------
def run_loop(instruction, max_steps=6):
    messages = [
        {"role": "system", "content": "你是车载座舱助手。真实状态必须调工具确认；"
                                       "工具结果才是事实；用户没给必要参数时先问，不要猜。"},
        {"role": "user", "content": instruction},
    ]
    for step in range(1, max_steps + 1):
        resp = chat(messages, [t["schema"] for t in TOOLS.values()])      # CORE 1 调模型
        msg = resp["choices"][0]["message"]
        calls = msg.get("tool_calls")
        # 白名单重建：别把 reasoning_content 之类塞回去，下一轮会报不认识的字段
        echoed = {"role": "assistant", "content": msg.get("content") or ""}
        if calls:
            echoed["tool_calls"] = calls
        messages.append(echoed)

        if not calls:                                                     # CORE 2 停止条件
            print(f"[step {step}] 结束：没有 tool_calls")
            return msg.get("content", "")

        for c in calls:                                                   # CORE 3 逐个执行
            name = c["function"]["name"]
            raw = c["function"]["arguments"] or "{}"
            try:
                args = json.loads(raw)
            except json.JSONDecodeError:
                result = {"ok": False, "error": {"code": "BAD_ARGUMENTS",
                                                 "hint": "重新给一段合法 JSON"}}
            else:
                tool = TOOLS.get(name)
                result = tool["handler"](args) if tool else {
                    "ok": False, "error": {"code": "UNKNOWN_TOOL", "hint": f"可用 {list(TOOLS)}"}}
            print(f"[step {step}] {name}({raw}) -> {json.dumps(result, ensure_ascii=False)[:110]}")
            messages.append({"role": "tool", "tool_call_id": c["id"],      # CORE 4 回灌
                             "content": json.dumps(result, ensure_ascii=False)})
    return "达到最大步数，未完成"

In [ ]:
# ---- 4/4 跑一次：看 STATE 怎么变，不要只看它说了什么 ---------------------
print("跑前:", STATE)
print("模型说:", run_loop("把空调调到 24 度，然后告诉我空调现在的状态"))
print("跑后:", STATE)   # 终态读 STATE，不读上一行那句话
print()
print("模型说:", run_loop("车里有点热，调凉一点"))
print("跑后:", STATE)   # 这次它应当拒绝猜数值

**真实跑通的一次（DeepSeek，不是 mock）：**

```
[step 1] ok  control_ac({"temperature": 24}) -> {"ok": true, "data": {"ac_temp": 24, "ac_on": true}}
[step 2] ok  get_vehicle_status({}) -> {"ok": true, "data": {"ac_temp": 24, "ac_on": true, "speed_kmh": 0}}
[step 3] 结束：没有 tool_calls

--- 结果 ---
reason=done steps=3 usage={'prompt_tokens': 2152, 'completion_tokens': 117, 'latency_s': 3.631, 'cost_yuan': 0.00524}
模型最后说：空调已调到 24 度。当前状态：
- 空调：已开启
- 目标温度：24 ℃
- 车速：0 km/h（车辆静止）
```

注意 step 2：**没人叫它读状态，它自己调了 `get_vehicle_status`**。这是 `description` 里那句
「在执行任何改动类操作之前……调用，用来确认真实状态」起的作用——工具描述是 prompt 的一部分。

第二次故意给模糊指令「车里有点热，调凉一点」，它的反应值得记：

```
[step 1] ok  get_vehicle_status({}) -> {"ok": true, "data": {"ac_temp": 26, "ac_on": false, "speed_kmh": 0}}
[step 2] 结束：没有 tool_calls
模型最后说：空调是关闭状态，目标温度 26℃……您想设成多少度？告诉我一个具体数值（16~30℃）。
```

它去读状态查清「为什么热」，然后**反问数值而不是猜 24 度**。这一条是
`control_ac` 的 `description` 里「用户只说调高一点而没有调数值时不要猜，先问清楚」写出来的，
不是模型天生就守规矩。这两次对比就是你 §3 要做的「重写 description」实验的起点。

另外我用 mock client 跑了 4 个场景证明链路：正常一轮、工具拒绝后带 `hint` 重试、未知工具、
`arguments` 是坏 JSON。四种都走通了，`traces/*.jsonl` 也真的落了盘。

**关掉文件后，这 5 件事必须能写出来（写不出就是没读懂）：**

1. `messages` 从哪两条开始，`system` 里放了什么
2. 模型响应里判断「结束」的唯一条件是什么
3. `arguments` 的类型是什么，`json.loads` 在哪一层
4. 工具失败时回灌的那条消息长什么样（`role` / `tool_call_id` / `content`）
5. 为什么 assistant 消息要白名单重建，而不是直接 `messages.append(msg)`

---

## 2. 踩坑（15 min）

拿 `control_ac(-273)` 打一次，看是谁在报错：你的 `run()`、`invoke()`、还是工具本身。

- 报错原文：
- 我猜的原因：
- 实际原因：

---

## 3. 今天踩的坑 / 意外发现

-

## 4. 明天开工前，我要先改的一件事

-